# TrafficFlowBench 2026 — Technical Report

**Competition:** `2026-ieee-big-data-traffic-flow-bench`
**Team:** CacinieP
**Best public leaderboard:** **0.86535** — submission ref `57668351` (2026-10-26), kernel `cacinie/tfb-v5orig` version 7
**Public LB position:** rank **224 / 660** teams as of the 2026-10-28 15:41 UTC leaderboard snapshot
(leader 0.92817, p10 0.88155, median 0.84810). Submitted through the Kaggle CLI, 5/day quota.
**Final submission kernel:** `cacinie/tfb-v5orig` (**public**, deterministic inference only;
public CPU re-run pushed as version 10 for verification)
**Diagnostic kernels:** `cacinie/tfb-queue-diag`, `cacinie/tfb-state-rootcause`, `cacinie/tfb-lwr-floor`, `cacinie/tfb-physics-diag`, `cacinie/tfb-queue-countmatch`, `cacinie/tfb-v16a-residual-stack-eval`
**Repository:** `CacinieP/2026-ieee-big-data-traffic-flow-bench` (private repo, code mirrored into the public Kaggle notebook)
**Date:** 2026-10-28

---

## 1. Contract and data understanding

Ten directional corridor panels (`D7_I10_E/W`, `D7_I210_E/W`, `D7_I405_N/S`, `D12_I5_N/S`,
`D12_I405_N/S`), 5-minute resolution, 273 released train days (2030-06-01 … 2031-02-28), a hidden
**validation** month (2031-03, public LB) and a hidden **private** month (2031-04, private LB, decides awards).

Total score is a weighted sum of four task scores (`docs/competition_contract.json`):

```text
S_total = 0.35*S_state + 0.30*S_queue + 0.15*S_physics + 0.20*S_ODME
```

| Task | What is scored | Metric |
|---|---|---|
| state (0.35) | reconstruct masked `speed_kmh` / `flow_vph` under regimes R1/R2/R3 = 20/30/50 % masking | `S = 0.54*(1 - RMSE_v/25) + 0.46*(1 - RMSE_q,per-lane/600)`, macro-mean over regimes |
| queue (0.30) | binary queue status for `T+5 … T+30` given `T-60 … T-5` history, 8 panels × 5 onset + 5 ongoing windows | space-time IoU per window, equal mean over conditions |
| physics (0.15) | no independent output — derived from the submitted state rows | `(1/3)*S_FD + (2/3)*S_LWR`, `S_LWR = 1 - Σ|dN-rhs|/Σ|rhs|` |
| ODME (0.20) | path flows | `0.45*S_od + 0.25*S_link + 0.15*S_dev + 0.15*S_attr` |

Three contract details drove our work:

1. **Eligibility is per cell** (`pct_observed >= 75`); flow is scored **per lane**, so a
   submission may carry total link flow and the evaluator divides by lane count.
2. **Queue truth is defined on the underlying state** — `speed <= v_cut`, `v_cut = 0.60 * free_speed`
   — not on the noisy observation, and both conditions are guaranteed to contain a queue in the horizon.
3. **Physics has no free parameters**: density is the identity `k = q/v`, accumulation `N = k*L`,
   boundary flows come from the released topology applied to *our own* flows, ramp flows from the
   released observations. Only the Task 1 answer can move `S_physics`.

### Final result at a glance

| item | value | how measured |
|---|---|---|
| best public LB | **0.86535** | submission ref 57668351 (2026-10-26) |
| public rank | **224 / 660** | leaderboard snapshot 2026-10-28 15:41 UTC |
| leader / p10 / median | 0.92817 / 0.88155 / 0.84810 | same snapshot |
| public score of the replicated recipe | 0.86517–0.86518 | public notebook / our ref 57545273 |
| our delta over the recipe | +0.00015 to +0.00018 | single change: queue ongoing threshold 0.40 → 0.50 |
| state / queue / physics / ODME (deconvolved) | ≈0.941 / 0.887 / ≈0.66 / ≈0.854 | §3.4 |

We deliberately never download the 8.3 GB release: every experiment runs as a Kaggle kernel, and the
local repository holds code, diagnostics and the submission ledger only.

## 2. Method per task

### 2.1 State reconstruction (weight 0.35)

A **hybrid** model that predicts a *residual* on top of an explicit interpolation baseline:

1. Per link, interpolate the observed speed/flow over time; for flow, average the temporal estimate
   with the mean observed flow of topology-adjacent links.
2. Predict two residuals with gradient-boosted trees: `r_v = v - v_0` and `r_q = q/L - q_0`
   (flow normalised per lane before the residual).
3. Final prediction = baseline + residual; flow is multiplied back by lane count and clipped.

Our production model is the published **0.86517** recipe's CatBoost ensemble: two depths
(d8, d10) per channel, averaged with weights **0.35 / 0.65** (the public recipe uses 0.5 / 0.5).
The 94 features are, in the recipe's own layout:

* temporal lags/leads at ±1, ±2, ±3, ±6, ±12 steps (speed, flow/lane, `q/v`, observation flag);
* nearest previous/next observed value of the same station and the two gap sizes;
* topology neighbours: mean speed/flow and counts over incoming/outgoing links, their differences;
* milepost-rank neighbours at ±1, ±2: same-instant observed value, temporally filled value,
  milepost distance;
* static and calendar context: panel id, link id, regime, `v_f`, capacity/lane, lanes, length,
  time-of-day (raw + sin/cos), day-of-week.

Measured on our own Feb-2031 holdout (420 k sampled target cells, the recipe's own masking code and seeds):
**speed RMSE 1.600 km/h**, **flow RMSE 32.17 vph per lane**, i.e.

```text
S_state = 0.54*(1 - 1.600/25) + 0.46*(1 - 32.17/600) = 0.9408
```

### 2.2 Queue forecasting (weight 0.30)

Two LightGBM/CatBoost classifiers on 60-minute causal history (36 and 83 features: last/lag values,
rolling means, queue age and counts, missingness, speed normalised by free-flow speed, flow by
capacity, milepost-neighbour and topology-neighbour values, time-of-day).

* **onset windows**: a LightGBM + CatBoost blend (weight 0.5, threshold **0.55**), and — this is a
  structural property of the task that we measured rather than assumed — the prediction is written
  **only at the last horizon step**. 97.5 % of the official onset windows have their entire queue
  truth at `T+30` (mean 3.52 cells); broadcasting the selection to all six steps collapses IoU from
  0.972 to 0.166.
* **ongoing windows**: independent per-step probabilities, thresholded at **0.50**. This is our
  **only deviation from the public recipe** (which uses 0.40); see §4.

Measured on the 80 official train windows (8 scored panels × 5 onset + 5 ongoing):
onset IoU **0.9719**, ongoing IoU **0.8013** at the recipe's threshold.

### 2.3 Physics (weight 0.15)

No output is submitted. We verified (§3.3) that the conservation term cannot be measured locally
without the organizer-held boundary flux, so we did not attempt to optimize it directly; the only
legitimate route is a better/more consistent state answer.

### 2.4 ODME (weight 0.20)

Non-negative ridge on path flows:
`min_{f>=0} 0.5*||A f - c||^2 + (λ/2)*||f - b||^2`, `A` the released path-link incidence, `c` the
released link counts, `b` the released weak prior. Production setting: **λ = 0.02** (the recipe ships
0.05), warm start shipped with the assets, solver tolerance tightened. Changing λ from 0.05 to 0.02,
rebuilding the warm start, and deepening solver convergence each moved the LB by ≤ 2e-5 (see §4),
so we froze this task.

## 3. Diagnostic chain — measuring before changing

All four axes are "saturated" in the sense that knob-turning does nothing. Rather than stop there we
measured **where the error actually lives**, because that decides which lever is worth building.

### 3.1 State error is diffuse, not concentrated in unsupported cells

Kernel `tfb-state-rootcause` reproduces the exact holdout (speed 1.60007 / flow 32.16866, identical
to the earlier `tfb-v16a` measurement) and decomposes the squared error by structures the model can
see. Shares are of total SSE:

| Slice | row share | speed RMSE | speed SSE | flow SSE |
|---|---|---|---|---|
| mask R1 (20 %) | 31.4 % | 1.4333 | 25.2 % | 29.9 % |
| mask R2 (30 %) | 34.3 % | 1.5223 | 31.0 % | 32.3 % |
| mask R3 (50 %) | 34.3 % | 1.8074 | 43.8 % | 37.8 % |
| own-station observation within ±1 step | 76.4 % | 1.5301 | **69.9 %** | 73.8 % |
| own-station observation within ±3 steps | 97.9 % | 1.5900 | **96.7 %** | 97.4 % |
| no spatial neighbour observation (`spatial_count = 0`) | 52.7 % | 1.7018 | 59.6 % | 57.3 % |
| temporal gap 2–3 steps | 47.7 % | 1.6088 | 48.3 % | 48.1 % |
| **neither temporal nor spatial support** | **0.03 %** | 1.726 | **0.03 %** | 0.04 % |

Counterfactuals (make one slice perfect, keep the rest):

| If this slice were perfect | speed RMSE | relative | flow RMSE | relative |
|---|---|---|---|---|
| all `spatial_count = 0` cells (52.7 % of rows) | 1.6001 → 1.4784 | **−7.6 %** | 32.17 → 30.57 | −5.0 % |
| all cells with temporal gap > 3 steps | 1.6001 → 1.5349 | −4.1 % | → 31.72 | −1.4 % |
| all cells with no temporal AND no spatial support | 1.6001 → 1.6000 | 0.0 % | → 32.17 | 0.0 % |
| the whole hardest regime R3 | 1.6001 → 1.4805 | −7.5 % | — | — |

**Conclusion.** 99.95 % of the error sits on cells that *do* have temporal and spatial support, and
the two strongest "perfect one slice" bounds are worth only −7.6 % RMSE ≈ **+0.0016 total score**.
Spatial-neighbour pooling — the direction we had registered as the next step — is therefore not the
lever: its ceiling is 0.002 points. The error is a diffuse noise floor.

### 3.2 Three cheap corrections, all below the pre-registered gate

Gate: both channels must improve ≥ 0.3 % relative on the Feb-2031 holdout.

| Candidate | speed | flow | gate |
|---|---|---|---|
| A. per (panel × regime) additive residual bias | 0.0037 % | 0.0024 % | ✗ |
| B. per (panel × gap-bin) additive residual bias | 0.0094 % | 0.0064 % | ✗ |
| C. HistGBDT on 15 error-structure columns (train months → Feb) | 0.0133 % | 0.0730 % | ✗ |
| D. (control) full-feature HistGBDT residual stack, kernel `tfb-v16a` | 0.223 % | 0.208 % | ✗ |

The best correction is 4–20× short of the gate. This is an information limit, not a tuning limit.

### 3.3 Physics cannot be measured offline (and therefore not gated)

The official LWR term needs organizer-held boundary flows, which are never published. Without them
the official scorer falls back to topology-derived flux, for which a mainline link's inflow and
outflow nearly cancel; `Σ|rhs|` becomes tiny and `E_LWR` saturates. Kernel `tfb-lwr-floor` measured
`E_LWR` for four fields whose quality we know, on the released Feb-2031 month:

| field | D7_I10_E | D12_I5_N |
|---|---|---|
| **the released observation field itself** | **1.0039** | **1.0004** |
| observations + 3-point temporal smoothing | 1.0013 | 1.0003 |
| observations with 30 % blanked then interpolated | 1.0182 | 1.0024 |
| observations with 50 % blanked then interpolated | 1.0145 | 1.0017 |

Even the best field that exists scores `E ≈ 1` (`S_LWR = 0`) under the fallback. Likewise kernel
`tfb-physics-diag` scored four submitted-field variants (raw, 3-point smoothing on both channels,
speed only, flow only) with the official scorer: `S_physics = 0.3293` for all four, Δ ≤ 7e-6.
**Physics is unmeasurable offline**; the axis can only be attempted as an un-gateable bet, so we froze it.

### 3.4 Decomposing the total, and what the headroom actually is

Substituting the measured values into the weighted sum:

```text
0.86520 - 0.35*0.9408 = 0.53593 = 0.30*S_queue + 0.15*S_physics + 0.20*S_ODME
```

If `S_physics` were the locally measured 0.329 the equation would require `S_ODME = 1.10 > 1`, which
is impossible — a consistency check that proves the official physics scale is much higher than the
local degenerate one. The self-consistent solution is approximately
`S_state ≈ 0.941, S_queue ≈ 0.887, S_physics ≈ 0.66, S_ODME ≈ 0.854`.

| task | current | value of a perfect score |
|---|---|---|
| state | 0.941 | +0.021 |
| queue | 0.887 | +0.034 |
| physics | ≈0.66 | +0.051 |
| ODME | ≈0.854 | +0.029 |

The gap to the public leader (0.92817 on 2026-10-28) is 0.0628 ≈ 47 % of the total remaining headroom:
the leaders are 10–20 % better on every axis rather than perfect on one. Our position (rank 224 of 660)
sits inside the densest part of the leaderboard — 244 teams score above 0.865 and 182 above 0.870, so
about 42 ranks correspond to +0.005 score at our operating point.

## 4. Single-variable experiment ledger (including negative results)

Every entry changed exactly one thing against the then-current best.

| # | submission ref | public LB | single variable | verdict |
|---|---|---|---|---|
| 1 | 56898465 | 0.55079 | (earlier lineage) temporal-bucket conditioning | superseded |
| 2 | 57545273 | 0.86518 | replicate the public 0.86517 recipe | new baseline |
| 3 | 57555868 | 0.86518 | ODME solver convergence depth | 0 |
| 4 | 57556248 | 0.86520 | ODME λ 0.05 → 0.02 | +2e-5 (noise) |
| 5 | 57657349 | 0.86520 | rebuild the ODME warm start | 0 |
| 6 | 57657660 | 0.86520 | state d8/d10 mix 0.5/0.5 → 0.35/0.65 | 0 |
| 7 | **57666444** | **0.85486** | **queue thresholds onset 0.55→0.45 and ongoing 0.40→0.20** | **−0.01034 (rolled back)** |
| 8 | **57668351** | **0.86535** | **queue *ongoing* threshold only: 0.40 → 0.50** | **+0.00015 (current best)** |
| 9 | 57668637 | 0.86257 | queue ongoing threshold 0.50 → 0.60 | −0.00278 (overshot, rolled back) |

**Threshold ladder measured on the public leaderboard** (all three submissions differ only in the
ongoing-threshold value; every other component is identical):

| ongoing threshold | public LB | delta vs recipe |
|---|---|---|
| 0.20 (with onset 0.45) | 0.85486 | −0.01034 |
| 0.40 (recipe value) | 0.86520 | — |
| **0.50 (ours)** | **0.86535** | **+0.00015** |
| 0.60 | 0.86257 | −0.00278 |

The curve is flat-topped at 0.40–0.50 (Δ = 1.5e-4, i.e. at the resolution of the LB) and falls on both
sides, which is why we stopped probing this axis: the remaining harvest along it is ~1e-4, below the
noise band of the metric.

Experiment 7 is the most instructive negative result: the thresholds chosen by an IoU sweep on the
**official train windows** were *worse* than the shipped ones on the validation month. Two effects
compound: (a) the train-window proxy derives truth from the observation layer while the official
truth uses the underlying state, and observation noise around `v_cut` biases the sweep toward lower
thresholds; (b) probability calibration drifts with the month, so a lower threshold becomes a false
positive flood. Experiment 8 isolated the same axis one variable at a time and moved it the other way;
experiment 9 bracketed the optimum, which sits at 0.45–0.55.

Also negative, and reported in §3: state residual stacking (`tfb-v16a`, −0.22 %/+0.21 %), three
bias-correction gates, and four physics smoothing variants.

## 5. Reproduction

### 5.1 Environment

* Kaggle notebooks, Python 3.11 image, **GPU: Nvidia T4** for the production run, CPU is sufficient
  for every diagnostic kernel. No internet access in any kernel.
* Inputs attached to every kernel:
  1. competition dataset `2026-ieee-big-data-traffic-flow-bench` (via `competition_sources`);
  2. public dataset `anubisyy/trafficflowbench-086517-public-assets` (the 0.86517 recipe's
     feature/training/inference code and its seven checkpoints, plus `manifest.json` with SHA-256
     checksums that the kernel verifies on load).

### 5.2 Submission kernel

`cacinie/tfb-v5orig` — notebook, **public**, deterministic inference (`RUN_TRAINING = False`,
`RUN_INFERENCE = True`), no external data beyond the two inputs above.

```bash
# 1. fetch the code (identical to the repository copy under kernels/tfb-v5orig/)
kaggle kernels pull cacinie/tfb-v5orig -p ./tfb-v5orig --metadata

# 2. or run it as-is on Kaggle: open the notebook, attach the public assets dataset, enable GPU, Run All
kaggle kernels status  cacinie/tfb-v5orig
kaggle kernels output  cacinie/tfb-v5orig -p ./out          # writes submission.csv

# 3. reproduce the submitted score
kaggle competitions submit 2026-ieee-big-data-traffic-flow-bench \
    -f ./out/submission.csv -m "reproduction of ref 57668351"
```

Expected behaviour: ≈ 25–30 min on a T4; prints per-panel row counts and the queue positive rate;
writes `submission.csv` with **6 980 503 rows** and the schema
`submission_id,task,speed_kmh,flow_vph,queue_pred,path_flow`, plus per-task intermediate files
(`predictions/state.csv`, `queue.csv`, `odme.csv`) and a run report. The kernel asserts the row count,
that no join key is missing, that speeds are positive and `queue_pred ∈ {0,1}`.

### 5.3 Diagnostic kernels (each reproduces one table above)

| kernel | what it reproduces | runtime |
|---|---|---|
| `cacinie/tfb-state-rootcause` | §3.1 decomposition + §3.2 gates A/B/C | ~25 min (GPU) |
| `cacinie/tfb-v16a-residual-stack-eval` | §3.2 row D | ~20 min (GPU) |
| `cacinie/tfb-queue-diag` | §2.2 IoU on the 80 official windows | ~5 min (CPU) |
| `cacinie/tfb-lwr-floor` | §3.3 fallback `E_LWR` table | ~10 min (CPU) |
| `cacinie/tfb-physics-diag` | §3.3 official-scorer variant table | ~30 min (CPU) |
| `cacinie/tfb-queue-countmatch` | §6 month-split count-matching gate | ~12 min (CPU) |

All diagnostics write a single JSON next to the log; the repository copies are in `kernels/output/`.

## 6. Limitations and what we would do with more time

* **The binding constraint is the absence of a local validation signal for the two largest axes.**
  Queue truth exists only for the hidden months and physics needs unpublished boundary flows; the
  train-window proxy for queue provably mis-transfers across months (experiment 7).
* The state model is at its information floor for this feature set: three independent correction
  attempts (0.004 %, 0.013 %, 0.22 %) all missed a 0.3 % gate, and the best "perfect one slice"
  counterfactual is worth +0.0016.
* **Queue count matching was tested and failed its gate.** Rule: per window, per step, mark the
  top-k links by probability with `k = round(rho * (queued links in the last observed history row))`,
  fitted on Nov/Dec/Jan (120 windows) and validated on Feb-2031 (40 windows), using windows we built
  ourselves because the official train windows cover a single month (2030-06). Result: fitted
  `rho = 1.15`, Feb IoU **0.5318 vs 0.6008** for a fixed 0.50 threshold (`-0.069`), and `rho` fitted
  per month spread 13.2 % (> the 10 % stability gate). Rejected without spending a submission.
  A useful by-product: this month-split proxy reproduces the LB ordering of the fixed-threshold curve
  (Feb: 0.4 → 0.6007, 0.5 → 0.6008, 0.6 → 0.5953, 0.2 → 0.5884; LB: 0.86520 / 0.86535 / 0.86257 /
  0.85486), so it is the first proxy that transfers and should be the basis for future queue work.
* Next experiments we would run, in order: (a) an occupancy/density information check — those shipped
  channels are the only ones the model does not consume, and they are likely a transform of `q/v`;
  (b) a month-robust queue rule (per-window count matching against the last observed queue count,
  fitted on Nov–Jan and validated on Feb) whose gate is "held-out-month IoU +0.005 with a stable
  ratio"; (c) if a way to observe conservation-consistent scoring appears, a state post-processing
  step projecting the answer onto the FD/conservation manifold while bounding the state RMSE loss.

## 7. Statement of compliance

* The submission was produced by the public notebook `cacinie/tfb-v5orig` from the competition data
  and one public dataset; no private data, no hand-labelling of targets, no leaderboard probing of
  hidden labels.
* All single-variable experiments, including the two that lost 0.0103 and 0.0028 points, are listed
  in §4 with their submission refs.
* The code is deterministic (inference only, fixed checkpoints, no training in the submission path).